# AI Attendance Manager - Model Training
This notebook trains a Convolutional Neural Network (CNN) for facial recognition.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle

### Data Pre-processing
Load images, resize, and normalize.

In [ ]:
dataset_path = 'dataset'
faces = []
labels = []

for student_folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, student_folder)
    if os.path.isdir(folder_path):
        student_id = student_folder.split('_')[0]
        for image_name in os.listdir(folder_path):
            image_path = os.path.join(folder_path, image_name)
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (128, 128))
                faces.append(img)
                labels.append(student_id)

faces = np.array(faces).reshape(-1, 128, 128, 1) / 255.0
print(f"Loaded {len(faces)} images.")

### Label Encoding and Data Splitting

In [ ]:
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded)

os.makedirs('models', exist_ok=True)
with open('models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

x_train, x_test, y_train, y_test = train_test_split(faces, labels_categorical, test_size=0.2, random_state=42)

### Build and Train CNN Model

In [ ]:
num_classes = len(le.classes_)
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 1)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test), batch_size=32)

In [ ]:
model.save('models/attendance_model.h5')
print("Model training complete and saved.")